In [124]:
import nltk
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('wordnet')
import re
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet as wn
import pandas as pd

import pandas as pd
import spacy
from spacy.training import Example

In [125]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1.8 MB/s eta 0:00:00a 0:00:01m

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3.11 install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


- en_core_web_sm — это одна из предобученных моделей для обработки естественного языка (NLP), предоставляемая библиотекой spaCy. Эта модель предназначена для работы с английским языком и включает в себя различные функции для анализа текста.

- Функции:

- Токенизация: Разделение текста на отдельные токены (слова, знаки препинания и т.д.).
- Разметка по частям речи (POS tagging): Определение частей речи для каждого токена (существительное, глагол, прилагательное и т.д.).
- Лемматизация: Приведение слов к их начальной форме (например, "running" к "run").
- Синтаксический анализ: Определение грамматической структуры предложения, включая зависимые отношения между словами.
- Распознавание именованных сущностей (NER): Выделение именованных сущностей, таких как имена людей, организации, даты и т.д.    

In [126]:
file_path = "/Users/MAC/Desktop/Компьютерная лингвистика/articles.txt"

def split_articles_by_title(file_path):
    with open(file_path, encoding="utf-8") as file:
         content = file.read()
    #разделяем содержимое по вхождениям "Title:" с сохранением этого слова в начале каждого блока
    articles = content.split("\nTitle:")
    
    #очищаем каждую статью от лишних пробелов и символов
    articles = [article.replace('\n', ' ').strip() for article in articles]
    
    return articles

articles = split_articles_by_title(file_path)

#выводим первую статью
print(articles[0])

Title: Combining Machine Learning and Computational Chemistry for Predictive Insights Into Chemical Systems Authors: Bingqing Cheng, Valentin Vassilev-Galindo, John A Keith, Alexandre Tkatchenko, Klaus-Robert Müller, Stefan Chmiela, Michael Gastegger Organizations:  †Department of Chemical and Petroleum Engineering Swanson School of Engineering, University of Pittsburgh, Pittsburgh, Pennsylvania 15261, United States,  ‡Department of Physics and Materials Science, University of Luxembourg, L-1511 Luxembourg City, Luxembourg,  ¶Accelerate Programme for Scientific Discovery, Department of Computer Science and Technology, 15 J. J. Thomson Avenue, Cambridge CB3 0FD, United Kingdom,  §Department of Software Engineering and Theoretical Computer Science, Technische Universität Berlin, 10587, Berlin, Germany,  §Department of Software Engineering and Theoretical Computer Science, Technische Universität Berlin, 10587, Berlin, Germany,  ∥Machine Learning Group, Technische Universität Berlin, 10587

In [127]:
#загружаем языковую модель.
nlp = spacy.load("en_core_web_sm")

In [130]:
#скачаем датасет для дообучения модели для распознавания сущностей
file_path = "/Users/MAC/Desktop/Компьютерная лингвистика/NER dataset.csv"
data = pd.read_csv(file_path, encoding='windows-1251')
data

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O
...,...,...,...,...
1048570,NaN,they,PRP,O
1048571,NaN,responded,VBD,O
1048572,NaN,to,TO,O
1048573,NaN,the,DT,O


In [133]:
data[data['Sentence #']== 'Sentence: 1']

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O


O: Это метка "Outside", которая указывает, что токен (слово) не является частью какой-либо именованной сущности. То есть, это слово не относится ни к одной из категорий


B-: Это префикс, который указывает на начало именованной сущности. Например:

B-geo: Начало географической сущности (например, название города или страны).
B-tim: Начало временной сущности (например, дата или время).
B-org: Начало организационной сущности (например, название компании или учреждения).
B-per: Начало персональной сущности (например, имя человека).
B-gpe: Начало геополитической сущности (например, название региона или страны).
B-art: Начало артефактной сущности (например, название произведения искусства).
B-eve: Начало событийной сущности (например, название события).
B-nat: Начало национальной сущности (например, название нации).


I-: Это префикс, который указывает на продолжение именованной сущности. Например:

I-per: Продолжение персональной сущности.
I-org: Продолжение организационной сущности.
I-geo: Продолжение географической сущности.
I-tim: Продолжение временной сущности.
I-gpe: Продолжение геополитической сущности.
I-art: Продолжение артефактной сущности.
I-eve: Продолжение событийной сущности.
I-nat: Продолжение национальной сущности.


##### обучаем модель

In [135]:
#преобразование данных в формат для обучения
train_data = []
current_sentence = []
current_entities = []


for index, row in data.iterrows():
    if pd.isna(row['Word']):
        #если слово пустое, это конец предложения
        if current_sentence: #проверяем, есть ли в current_sentence слова
            train_data.append((current_sentence, {'entities': current_entities}))
            current_sentence = []
            current_entities = []
    else:
        #добавляем слово в текущее предложение
        current_sentence.append(row['Word'])
        if row['Tag'] != 'O':
            #если тег не 'O', добавляем его как сущность
            start = len(' '.join(current_sentence)) - len(row['Word'])
            end = start + len(row['Word'])
            current_entities.append((start, end, row['Tag']))

#обработка последнего предложения, если оно не завершено
if current_sentence:
    train_data.append((current_sentence, {'entities': current_entities}))




# Преобразуем в формат (текст, аннотации)
train_data = [(' '.join(sentence), annotations) for sentence, annotations in train_data]


#обучение модели
for epoch in range(10):  #количество эпох
    for text, annotations in train_data:
        #проверка длины текста
        if len(text) > 200000:
            text = text[:200000]  #обрезаем текст до 200 000 символов для ускорения обучения
        doc = nlp.make_doc(text)#метод make_doc используется для создания объекта doc из строки text. Этот объект представляет собой документ, который будет использоваться для обучения модели. Он содержит токены, которые были созданы из текста. 
        example = Example.from_dict(doc, annotations)
        '''Здесь создается объект example с помощью метода from_dict, который принимает документ doc и аннотации annotations. 
           Этот объект представляет собой пример, который будет использоваться для обновления модели. 
           Он содержит информацию о том, какие токены в документе соответствуют каким сущностям. '''
        nlp.update([example])

#сохранение дообученной модели
nlp.to_disk("my_precious")


#загрузка дообученной модели
nlp_custom = spacy.load("my_precious")


#функция для распознавания сущностей в тексте
def recognize_entities(text):
    doc = nlp_custom(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]   
    return entities


#функция для фильтрации имен и организаций
def filter_entities(entities):
    filtered = []
    for text, label in entities:
        #проверяем, содержит ли текст число
        if re.search(r'\d', text):
            continue  #пропускаем сущности с числами

        #исключаем сущности с более чем одной заглавной буквой
        if sum(1 for char in text if char.isupper()) > 1:
            continue

        if label in ['B-per', 'I-per']:  #фильтруем только имена и фамилии
            filtered.append((text, label))
    return filtered


#функция для склеивания сущностей и удаления дубликатов
def merge_entities(entities):
    merged_names = set()  #для имен
    for text, label in entities:
        if label in ['B-per', 'I-per']:
            merged_names.add(text)  #добавляем имена и фимилии

    #преобразуем множества обратно в строки
    return {
        'names': ', '.join(merged_names)
    }



#обработка статей
for i, article in enumerate(articles):
    print(f"Статья {i + 1}:")
    entities = recognize_entities(article)
    filtered_entities = filter_entities(entities)
    merged_entities = merge_entities(filtered_entities)



    print("Имена и фамилии:", merged_entities['names'])
    print("\n")

Статья 1:
Имена и фамилии: de, K., minima, symposia, Hyperparameter, Dral, Gastegger, Artrith, Bernstein, ,, Burke, Note, They, Waals, Klucznik, Griego, Atomistic, Θ., Carlo, Kernel, Slater, Investigator, Hermann, Molecules, P., Zhong, Müller, Fermions, Ii, Parinello, A., Gerhard, Exc, Dirac, Ĥel, George, Slater-type, Research, σ(s, Mater, H., kijk, Wu, al, S., Vrep, Stefan, Infrared, F, Computer, Keith, Hilbert, Usually, Ref, i+n, Optimizations, Fock, G., Miller, Professor, Copyright, Lagrange, Wasserstein, T., Bingqing, van, Algorithms, Virtual, von, ref, Chmiela, Al, Brockherde, Alexander, Rappe, Math+, Generally, Parrinello, Screening, Charles, John, Grant, Minima, Koster, N., Raman, A, Hall, Boltzmann, ; Hirzel, Harrison, −, Michael, Alexandre, Ertl, Multiscale, D., Tkatchenko, Bag, Medal, Predictive, −Π, Faculty, Humboldt, Merino, J., Factorization, Monte, kinase, –, Vatt, Fellow, Coulomb, Combining, Box, e.g., Hammett, Review, Efficient, Nature, Lilienfeld, Standard, Zhavoronkov